# Day 6 — PySpark Practice Questions: Window Functions Part 1

**3 questions** · Easy → Medium  
Solutions are in hidden cells below each question — try first!

In [ ]:
import os
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day6_Window_Practice') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

# Same data as SQL — Bob & Carol tie in North/month1; Frank & Grace tie in South/month1
sales_data = [
    (1, 'Alice', 'North', 1, 5000.0),
    (2, 'Bob',   'North', 1, 4200.0),
    (3, 'Carol', 'North', 1, 4200.0),
    (4, 'Dave',  'North', 1, 3800.0),
    (5, 'Eve',   'South', 1, 6100.0),
    (6, 'Frank', 'South', 1, 5500.0),
    (7, 'Grace', 'South', 1, 5500.0),
    (8, 'Hank',  'South', 1, 4000.0),
    (1, 'Alice', 'North', 2, 5300.0),
    (2, 'Bob',   'North', 2, 4800.0),
    (3, 'Carol', 'North', 2, 4000.0),
    (5, 'Eve',   'South', 2, 5900.0),
    (6, 'Frank', 'South', 2, 6300.0),
    (7, 'Grace', 'South', 2, 5100.0),
]
schema = StructType([
    StructField('emp_id',   IntegerType(), False),
    StructField('emp_name', StringType(),  True),
    StructField('region',   StringType(),  True),
    StructField('month',    IntegerType(), True),
    StructField('revenue',  DoubleType(),  True),
])
df_sales = spark.createDataFrame(sales_data, schema=schema)
print('Rows:', df_sales.count())
df_sales.orderBy('region', 'month', F.desc('revenue')).show()

---
## Q1 (Easy) — Rank Employees by Revenue Within Each Region

Add a `revenue_rank` column that ranks every row by revenue **within each region** (all months together).  
Use `F.rank()`. Store result in `df_q1`, ordered by `region`, `revenue_rank`.

**Warm-ups:**
- What import is needed for `Window`?
- Write the Window spec: partition by region, order by revenue descending
- What method attaches the window function to the DataFrame?

In [ ]:
df_q1 = None
# YOUR CODE HERE

In [ ]:
# --- assert ---
assert df_q1 is not None, 'df_q1 not set'
assert 'revenue_rank' in df_q1.columns, 'revenue_rank column missing'

# Alice has the highest revenue in North (5300 in month 2) → rank 1
alice_north = df_q1.filter(
    (F.col('emp_name') == 'Alice') & (F.col('region') == 'North') & (F.col('month') == 2)
).collect()[0]
assert alice_north['revenue_rank'] == 1, f'Alice should be rank 1 in North, got {alice_north["revenue_rank"]}'

# Bob and Carol both have 4200 in month1 — they should have the same rank
bob  = df_q1.filter((F.col('emp_name') == 'Bob')   & (F.col('month') == 1)).collect()[0]
carol = df_q1.filter((F.col('emp_name') == 'Carol') & (F.col('month') == 1)).collect()[0]
assert bob['revenue_rank'] == carol['revenue_rank'], \
    f'Bob and Carol tie — should have same rank. Bob={bob["revenue_rank"]}, Carol={carol["revenue_rank"]}'

print('Q1 passed!')
df_q1.show()

### Solution

In [ ]:
# SOLUTION
w = Window.partitionBy('region').orderBy(F.desc('revenue'))

df_q1 = (
    df_sales
    .withColumn('revenue_rank', F.rank().over(w))
    .orderBy('region', 'revenue_rank')
)
df_q1.show()

---
## Q2 (Medium) — Top 2 Employees per Region, Month 1 Only (Ties Included)

Filter to `month == 1`. Within each region, find employees with `dense_rank <= 2`.  
Store in `df_q2`. Both tied employees at rank 2 must appear.

**Expected:**  
- North: Alice(1), Bob(2), Carol(2) → 3 rows  
- South: Eve(1), Frank(2), Grace(2) → 3 rows  
- Total: 6 rows

**Warm-ups:**
- Why `dense_rank` instead of `rank` for `<= 2` filtering?
- Do you filter before or after adding the window column?
- What is the type of `F.dense_rank().over(w)`?

In [ ]:
df_q2 = None
# YOUR CODE HERE

In [ ]:
# --- assert ---
assert df_q2 is not None, 'df_q2 not set'
assert df_q2.count() == 6, f'Expected 6 rows (3 per region), got {df_q2.count()}'

# All rows should be from month 1
assert df_q2.filter(F.col('month') != 1).count() == 0, 'Found rows not in month 1'

# Dave (rank 3 in North/month1) must NOT be in the result
dave_rows = df_q2.filter(F.col('emp_name') == 'Dave').count()
assert dave_rows == 0, f'Dave should not be in top 2 — got {dave_rows} rows'

# Bob AND Carol must both be present (tie at rank 2)
bob_rows   = df_q2.filter(F.col('emp_name') == 'Bob').count()
carol_rows = df_q2.filter(F.col('emp_name') == 'Carol').count()
assert bob_rows == 1 and carol_rows == 1, \
    f'Bob and Carol should both be in result. Bob={bob_rows}, Carol={carol_rows}'

print('Q2 passed!')
df_q2.orderBy('region', F.desc('revenue')).show()

### Solution

In [ ]:
# SOLUTION
w = Window.partitionBy('region').orderBy(F.desc('revenue'))

df_q2 = (
    df_sales
    .filter(F.col('month') == 1)
    .withColumn('drnk', F.dense_rank().over(w))
    .filter(F.col('drnk') <= 2)
    .drop('drnk')
    .orderBy('region', F.desc('revenue'))
)
df_q2.show()

---
## Q3 (Medium) — Employees Who Ranked #1 in Any Region for Any Month

Partition by **both** `region` and `month`. Find every row where `rank() == 1`.  
Return distinct `(emp_id, emp_name, region, month, revenue)` — no duplicates.

Store in `df_q3`, ordered by `region`, `month`.

**Expected:**  
- Alice: North/month1 and North/month2  
- Eve: South/month1  
- Frank: South/month2  
→ 4 rows total

**Warm-ups:**
- How do you partition by two columns: region AND month?
- If Eve and someone tied for #1 in South/month1, would both appear?
- Should you `.distinct()` the result? Why or why not?

In [ ]:
df_q3 = None
# YOUR CODE HERE
# Window.partitionBy('region', 'month').orderBy(F.desc('revenue'))

In [ ]:
# --- assert ---
assert df_q3 is not None, 'df_q3 not set'
assert df_q3.count() == 4, f'Expected 4 rows, got {df_q3.count()}'

# Expected (emp_name, region, month) combos
expected = {('Alice','North',1), ('Alice','North',2), ('Eve','South',1), ('Frank','South',2)}
actual   = {(r['emp_name'], r['region'], r['month']) for r in df_q3.collect()}
assert actual == expected, f'Wrong combinations. Got: {actual}'

print('Q3 passed!')
df_q3.orderBy('region', 'month').show()

### Solution

In [ ]:
# SOLUTION
w = Window.partitionBy('region', 'month').orderBy(F.desc('revenue'))

df_q3 = (
    df_sales
    .withColumn('rnk', F.rank().over(w))
    .filter(F.col('rnk') == 1)
    .select('emp_id', 'emp_name', 'region', 'month', 'revenue')
    .distinct()
    .orderBy('region', 'month')
)
df_q3.show()

In [ ]:
spark.stop()
print('Spark stopped.')